# Experiment No. 09 — Personalized AI Assistant using Open-Source LLM, RAG, and LoRA/QLoRA

**Aim:** Develop a personalized AI assistant using an open-source foundational language model, retrieve answers from user-specific documents using Retrieval-Augmented Generation (RAG), and investigate how parameter-efficient fine-tuning affects response quality.

**Environment:** Google Colab (GPU recommended; T4 is sufficient for the small model used here).

### What this notebook does
1. Installs the required libraries and checks the runtime.
2. Loads an open-source instruction model (`Qwen/Qwen2.5-0.5B-Instruct`).
3. Creates a small example knowledge base, then optionally lets you upload your own PDF/TXT/DOCX files.
4. Chunks documents, embeds them, and retrieves relevant passages with FAISS.
5. Generates answers grounded in retrieved context.
6. Creates a tiny instruction dataset and fine-tunes a LoRA adapter.
7. Compares base-model, RAG, and LoRA+RAG responses.
8. Saves the adapter and vector index for download.

> **Important:** Run cells from top to bottom. Set **Runtime → Change runtime type → T4 GPU** for faster generation/fine-tuning. The notebook is designed to run with a small model, but Colab resource availability and package/model downloads can vary. Fine-tuning is optional if your session has insufficient memory. Do not upload confidential documents.

## 1. Install dependencies

In [ ]:
!pip -q install -U transformers accelerate sentence-transformers faiss-cpu peft datasets trl pypdf python-docx

## 2. Check runtime and configure settings

In [ ]:
import os, json, textwrap, random, torch
from pathlib import Path

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. RAG will work, but LLM generation and LoRA training may be slow.")

## 3. Load the open-source instruction model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True
)
if DEVICE == "cpu":
    model.to("cpu")
model.eval()

print("Model loaded:", MODEL_NAME)
print("Tokenizer vocabulary size:", len(tokenizer))

## 4. Create a sample personal knowledge base (works without uploading files)

In [ ]:
sample_docs = [
    {
        "source": "profile.txt",
        "text": """
        Personal Profile
        The student's name is Alex. Alex is a final-year computer science student.
        Alex is interested in generative AI, data engineering, Python, SQL, and machine learning.
        Alex's current project is a personalized AI assistant using an open-source language model,
        retrieval-augmented generation, and parameter-efficient fine-tuning.
        """
    },
    {
        "source": "project_notes.txt",
        "text": """
        Project Plan
        The assistant uses a language model to generate answers. Documents are split into smaller
        chunks and converted into vector embeddings. FAISS searches for chunks relevant to a query.
        The retrieved text is placed in the prompt as context. LoRA trains a small set of adapter
        parameters rather than updating every model parameter. QLoRA combines quantization with LoRA
        to reduce memory requirements. The project compares a base model, RAG, and LoRA-enhanced RAG.
        """
    },
    {
        "source": "study_notes.txt",
        "text": """
        Study Preferences
        The learner prefers step-by-step explanations, runnable Python examples, and clear outputs.
        Answers about uploaded documents should be based on the retrieved document content. If the
        documents do not contain an answer, the assistant should say that the information was not found.
        """
    }
]
print("Sample documents:", len(sample_docs))
for d in sample_docs:
    print("-", d["source"], "| characters:", len(d["text"]))

## 5. Optional: upload your own documents

In [ ]:
# Run this cell if you want to use your own PDF, TXT, or DOCX files.
# If no files are uploaded, the built-in sample documents remain available.
from google.colab import files

uploaded_docs = []
try:
    uploaded = files.upload()
except Exception:
    uploaded = {}

def read_uploaded_file(filename, content):
    suffix = Path(filename).suffix.lower()
    if suffix == ".txt":
        return content.decode("utf-8", errors="ignore")
    if suffix == ".pdf":
        from pypdf import PdfReader
        import io
        reader = PdfReader(io.BytesIO(content))
        return "\\n".join(page.extract_text() or "" for page in reader.pages)
    if suffix == ".docx":
        from docx import Document
        import io
        doc = Document(io.BytesIO(content))
        return "\\n".join(p.text for p in doc.paragraphs)
    print(f"Skipping unsupported file: {filename}")
    return ""

for filename, content in uploaded.items():
    extracted = read_uploaded_file(filename, content)
    if extracted.strip():
        uploaded_docs.append({"source": filename, "text": extracted})
        print(f"Read {filename}: {len(extracted)} characters")

if uploaded_docs:
    documents = uploaded_docs
    print("Using uploaded documents:", len(documents))
else:
    documents = sample_docs
    print("Using built-in sample documents:", len(documents))

## 6. Split documents into chunks

In [ ]:
def chunk_text(text, chunk_size=700, overlap=120):
    text = " ".join(text.split())
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

chunks, chunk_sources = [], []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append(chunk)
        chunk_sources.append(doc["source"])

print(f"Created {len(chunks)} chunks from {len(documents)} document(s).")
print("\nExample chunk:\n", chunks[0][:500] if chunks else "No text extracted.")

## 7. Build the embedding model and FAISS vector index

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

embeddings = embedder.encode(
    chunks, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine similarity because vectors are normalized
index.add(embeddings)

print("Embedding matrix shape:", embeddings.shape)
print("FAISS vectors indexed:", index.ntotal)

## 8. Retrieve relevant chunks

In [ ]:
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(query_vec, min(top_k, len(chunks)))
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            results.append({
                "score": float(score),
                "source": chunk_sources[idx],
                "text": chunks[idx]
            })
    return results

test_query = "What is the purpose of RAG and how does FAISS help?"
retrieved = retrieve(test_query, top_k=3)
print("Query:", test_query)
for i, item in enumerate(retrieved, 1):
    print(f"\n[{i}] source={item['source']} | similarity={item['score']:.3f}")
    print(item["text"][:350])

## 9. Generate answers with retrieved context

In [ ]:
def generate_text(prompt, max_new_tokens=180, temperature=0.2):
    messages = [
        {"role": "system", "content": "You are a helpful personal AI assistant. Be accurate and concise."},
        {"role": "user", "content": prompt}
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt")
    if DEVICE == "cuda":
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to("cpu") for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 0.01),
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def answer_with_rag(question, top_k=3):
    hits = retrieve(question, top_k=top_k)
    context = "\\n\\n".join(
        f"[Source: {h['source']}] {h['text']}" for h in hits
    )
    prompt = f"""
Answer the question using only the context below. If the context does not contain the answer,
say: "I could not find that information in the provided documents."
Do not invent personal facts.

Context:
{context}

Question: {question}

Answer:
"""
    answer = generate_text(prompt)
    return answer, hits

question = "What does this project use FAISS for?"
rag_answer, rag_hits = answer_with_rag(question)
print("QUESTION:", question)
print("\nRAG ANSWER:\n", rag_answer)
print("\nSOURCES USED:", sorted(set(h["source"] for h in rag_hits)))

## 10. Baseline: ask the model without document retrieval

In [ ]:
def answer_without_rag(question):
    prompt = f"Answer the following question briefly and do not claim access to private documents.\\nQuestion: {question}\\nAnswer:"
    return generate_text(prompt)

baseline_question = "According to my project notes, what does FAISS do?"
baseline_answer = answer_without_rag(baseline_question)
print("QUESTION:", baseline_question)
print("\nBASE MODEL ANSWER (no retrieval):\n", baseline_answer)
print("\nCompare this with the RAG answer above. The base model may not know facts specific to your documents.")

## 11. Prepare a small instruction dataset for LoRA fine-tuning

In [ ]:
# This is a tiny demonstration dataset. For a meaningful experiment, create more high-quality
# examples based on your own documents. Do not put sensitive data into training examples.
training_examples = [
    {
        "instruction": "What is the role of FAISS in this project?",
        "context": "FAISS searches a vector index to find document chunks relevant to the user's question.",
        "response": "FAISS retrieves the most relevant document chunks by comparing vector embeddings to the query embedding."
    },
    {
        "instruction": "Explain RAG in this project.",
        "context": "RAG retrieves relevant chunks from documents and includes them in the language model prompt.",
        "response": "RAG first retrieves relevant document passages, then supplies them as context so the language model can answer using the user's documents."
    },
    {
        "instruction": "What is LoRA?",
        "context": "LoRA trains a small set of adapter parameters instead of updating all model parameters.",
        "response": "LoRA is parameter-efficient fine-tuning that learns small adapter matrices while leaving most of the original model weights frozen."
    },
    {
        "instruction": "What is QLoRA?",
        "context": "QLoRA combines a quantized base model with LoRA adapters to reduce fine-tuning memory use.",
        "response": "QLoRA quantizes the base model and trains LoRA adapters, reducing memory requirements during fine-tuning."
    },
    {
        "instruction": "What should the assistant do when documents do not contain an answer?",
        "context": "The assistant should say that the information was not found rather than inventing an answer.",
        "response": "It should clearly say that the information could not be found in the provided documents and avoid making up facts."
    },
    {
        "instruction": "How are documents prepared for retrieval?",
        "context": "Text is extracted, split into overlapping chunks, embedded, and added to a FAISS index.",
        "response": "Extract the document text, split it into overlapping chunks, encode each chunk as an embedding, and add the vectors to the FAISS index."
    }
]

def format_training_example(ex):
    return (
        "### Instruction:\n" + ex["instruction"] +
        "\n\n### Context:\n" + ex["context"] +
        "\n\n### Response:\n" + ex["response"] + tokenizer.eos_token
    )

for ex in training_examples[:2]:
    print(format_training_example(ex), "\n---")

print("Training examples:", len(training_examples))

## 12. Fine-tune LoRA adapter (optional, GPU strongly recommended)

In [ ]:
# If training fails because of memory/runtime limits, skip this section and continue to Section 14.
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Keep a clean base-model copy for LoRA. This notebook fine-tunes the model loaded above.
# For this small model, standard LoRA is used; QLoRA requires bitsandbytes quantization setup.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

train_texts = [format_training_example(ex) for ex in training_examples]
train_dataset = Dataset.from_dict({"text": train_texts})

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=384)

tokenized_dataset = train_dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="/content/lora-personal-assistant",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=(DEVICE == "cuda"),
    use_cpu=(DEVICE == "cpu"),
    optim="adamw_torch",
    gradient_checkpointing=False,
    save_total_limit=2
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)
train_result = trainer.train()
print("Training complete.")
print("Training metrics:", train_result.metrics)

## 13. Save the LoRA adapter

In [ ]:
ADAPTER_DIR = "/content/personal-assistant-lora-adapter"
lora_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to:", ADAPTER_DIR)

## 14. Compare RAG before and after fine-tuning

In [ ]:
# If Section 12 completed, lora_model is available. Use the adapter model for generation.
# This helper uses the same tokenizer and chat template but runs whichever model is supplied.
def generate_with_model(active_model, prompt, max_new_tokens=180):
    messages = [
        {"role": "system", "content": "You are a helpful personal AI assistant. Use provided context accurately."},
        {"role": "user", "content": prompt}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt")
    target_device = next(active_model.parameters()).device
    inputs = {k: v.to(target_device) for k, v in inputs.items()}
    active_model.eval()
    with torch.no_grad():
        out = active_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.08, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def answer_with_rag_model(question, active_model, top_k=3):
    hits = retrieve(question, top_k)
    context = "\\n\\n".join(f"[Source: {h['source']}] {h['text']}" for h in hits)
    prompt = f"""Answer using only the context. If the answer is absent, say you could not find it.
Context:
{context}

Question: {question}
Answer:"""
    return generate_with_model(active_model, prompt), hits

comparison_question = "What is the difference between LoRA and QLoRA?"
before_answer, _ = answer_with_rag_model(comparison_question, model)
print("RAG BEFORE FINE-TUNING:\\n", before_answer)

try:
    after_answer, _ = answer_with_rag_model(comparison_question, lora_model)
    print("\\nRAG AFTER LoRA FINE-TUNING:\\n", after_answer)
except NameError:
    print("\\nLoRA training was skipped, so the fine-tuned comparison is unavailable.")

## 15. Simple evaluation checklist

In [ ]:
evaluation_questions = [
    "What does FAISS do?",
    "How does RAG use the documents?",
    "What is LoRA?",
    "What is QLoRA?",
    "What does the assistant do if the answer is missing?"
]

evaluation_rows = []
for q in evaluation_questions:
    hits = retrieve(q, top_k=3)
    evaluation_rows.append({
        "question": q,
        "retrieved_sources": ", ".join(sorted(set(h["source"] for h in hits))),
        "top_similarity": round(hits[0]["score"], 3) if hits else None
    })

import pandas as pd
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
print("Interpretation: retrieval similarity is a retrieval signal, not a complete measure of answer correctness.")

## 16. Save the vector index and document metadata

In [ ]:
import pickle

INDEX_DIR = "/content/personal_assistant_rag"
os.makedirs(INDEX_DIR, exist_ok=True)
faiss.write_index(index, os.path.join(INDEX_DIR, "documents.faiss"))

with open(os.path.join(INDEX_DIR, "metadata.pkl"), "wb") as f:
    pickle.dump({"chunks": chunks, "sources": chunk_sources, "embedding_model": EMBED_MODEL_NAME}, f)

print("Saved FAISS index and metadata in:", INDEX_DIR)

## 17. Download project artifacts

In [ ]:
# Download the adapter as a ZIP if LoRA training completed.
# Download the vector index and metadata separately as a ZIP.
import shutil

if os.path.exists("/content/personal-assistant-lora-adapter"):
    shutil.make_archive("/content/personal_assistant_lora_adapter", "zip", "/content/personal-assistant-lora-adapter")
    print("LoRA adapter ZIP: /content/personal_assistant_lora_adapter.zip")

shutil.make_archive("/content/personal_assistant_rag_files", "zip", INDEX_DIR)
print("RAG index ZIP: /content/personal_assistant_rag_files.zip")

# Uncomment to download to your computer:
# from google.colab import files
# files.download("/content/personal_assistant_rag_files.zip")
# files.download("/content/personal_assistant_lora_adapter.zip")  # if it exists

## 18. Conclusion and expected observations

- **Base LLM:** Can answer general questions, but may not know facts contained only in private documents.
- **RAG:** Retrieves document passages and supplies them as context, helping answers reflect the uploaded material and cite source filenames.
- **LoRA:** Updates a small number of adapter parameters and may improve the desired response format or task behavior when trained on enough good examples.
- **Limitations:** This notebook's six training examples demonstrate the workflow only. They are too few to establish a reliable quality improvement. Compare outputs across a larger held-out test set and use human scoring for correctness, relevance, groundedness, and completeness. Fine-tuning does not replace retrieval for frequently changing personal information.

### Troubleshooting
- **CUDA out of memory:** Restart runtime, use a smaller batch size, reduce `max_length`, or skip LoRA training.
- **No GPU:** RAG indexing usually works on CPU, but LLM generation/training may be slow. Enable a GPU in Runtime settings.
- **PDF extraction returns empty text:** The PDF may be scanned; OCR is needed before indexing.
- **Model download/authentication issue:** Check internet access in Colab and rerun the model-loading cell.
- **Need true QLoRA:** Install `bitsandbytes`, load the model with 4-bit `BitsAndBytesConfig`, then attach the same PEFT LoRA config. The main notebook uses ordinary LoRA for compatibility and clarity.